# Coordinate-Wise Length Smoke Test

This notebook checks the reviewer-requested coordinate-wise region length outputs. Run it from the repository root.

In [5]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "utility").exists():
    raise RuntimeError(f"Run this notebook from the repository root, not {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

Repo root: e:\multi-target-scaling


In [10]:
from utility.exps import run_abs_res_synthetic_experiment

methods = ["TSCP_R", "Empirical_copula", "Unscaled", "Point_CHR"]

result = run_abs_res_synthetic_experiment(
    dim_list=[4],
    sample_list=[30],
    alpha_list=[0.2],
    noise_type="Gaussian",
    trials=200,
    methods=methods,
    n_train_pool=8000,
    n_features=6,
    n_informative=6,
    oracle_n_samples=200,
)

result.summary_results

,alpha,n_dim,n_cal,method,noise_type,score_type,n_trials,test_coverage_avg,test_coverage_1std,coverage_vol_avg,coverage_vol_1std,coverage_max_length_median,runtime_avg
0,0.2,4,30,Empirical_copula,Gaussian,absolute_residual,200,0.740622,0.085562,303.536860,120.465664,7.479425,0.000395
1,0.2,4,30,Point_CHR,Gaussian,absolute_residual,200,0.807728,0.098037,602.215231,495.345566,8.576428,0.000555
2,0.2,4,30,TSCP_R,Gaussian,absolute_residual,200,0.810853,0.070919,415.083426,162.240075,8.041938,0.001121
3,0.2,4,30,Unscaled,Gaussian,absolute_residual,200,0.805781,0.067074,1266.003879,591.238340,5.817048,0.000070


In [11]:
coord_trial = result.coordinate_trial_results
coord_summary = result.coordinate_summary_results

assert not coord_trial.empty
assert not coord_summary.empty
assert set(coord_summary["method"]) == set(methods)
assert set(coord_summary["coordinate"]) == {1, 2, 3, 4}
assert (coord_trial["coordinate_length"] >= 0).all()

expected_trial_rows = 1 * 1 * 1 * 200 * len(methods) * 4
assert len(coord_trial) == expected_trial_rows

coord_summary.sort_values(["method", "coordinate"])

,alpha,n_dim,n_cal,method,noise_type,coordinate,score_type,n_trials,coordinate_length_avg,coordinate_length_1std,coordinate_length_median
0,0.2,4,30,Empirical_copula,Gaussian,1,absolute_residual,200,7.526754,1.421220,7.417498
1,0.2,4,30,Empirical_copula,Gaussian,2,absolute_residual,200,5.679626,0.965307,5.519698
2,0.2,4,30,Empirical_copula,Gaussian,3,absolute_residual,200,3.698191,0.646382,3.647559
3,0.2,4,30,Empirical_copula,Gaussian,4,absolute_residual,200,1.893811,0.333722,1.900224
4,0.2,4,30,Point_CHR,Gaussian,1,absolute_residual,200,8.802563,2.612722,8.434126
5,0.2,4,30,Point_CHR,Gaussian,2,absolute_residual,200,6.474953,1.762631,6.093511
6,0.2,4,30,Point_CHR,Gaussian,3,absolute_residual,200,4.345308,1.354423,4.129358
7,0.2,4,30,Point_CHR,Gaussian,4,absolute_residual,200,2.237510,0.593796,2.158771
8,0.2,4,30,TSCP_R,Gaussian,1,absolute_residual,200,8.092110,1.281599,7.986458
9,0.2,4,30,TSCP_R,Gaussian,2,absolute_residual,200,6.084959,0.854541,6.018495


In [12]:
wide_lengths = coord_summary.pivot_table(
    index="coordinate",
    columns="method",
    values="coordinate_length_avg",
)

wide_lengths

method,Empirical_copula,Point_CHR,TSCP_R,Unscaled
coordinate,,,,
1,7.526754,8.802563,8.092110,5.851544
2,5.679626,6.474953,6.084959,5.851544
3,3.698191,4.345308,4.035526,5.851544
4,1.893811,2.237510,2.038619,5.851544


Interpretation check: in the default synthetic setup, coordinate noise levels are `[d, d-1, ..., 1]`. A coordinate-adaptive method should usually show larger average lengths for noisier early coordinates and smaller lengths for quieter later coordinates, while less adaptive methods may look flatter across coordinates.